<a href="https://colab.research.google.com/github/FarahBelghith2/PFE/blob/main/01_database_to_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

backup_recette.sql
→ extraction des matchs/réservations
→ création des colonnes attendues par le modèle
→ CSV final compatible modèle

In [ ]:
import pandas as pd
import numpy as np
import re
from datetime import datetime
import requests
import ast
import time

In [ ]:
from google.colab import drive
drive.mount('/content/drive')




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:

SQL_PATH = "/content/drive/MyDrive/PFE/backup_recette.sql"
OUTPUT_RAW = "/content/drive/MyDrive/PFE/matches_extracted.csv"
OUTPUT_MODEL = "/content/drive/MyDrive/PFE/donnees_pretes_modele.csv"


In [ ]:
def extract_copy_table_from_sql(sql_path, table_name):
    with open(sql_path, "r", encoding="utf-8", errors="ignore") as f:
        sql = f.read()

    pattern = rf"COPY public\.{table_name} \((.*?)\) FROM stdin;\n(.*?)\n\\\."
    match = re.search(pattern, sql, flags=re.S)

    if not match:
        raise ValueError(f"Table {table_name} introuvable dans le dump SQL")

    columns = [c.strip() for c in match.group(1).split(",")]
    data = match.group(2).splitlines()

    rows = []
    for line in data:
        rows.append(line.split("\t"))

    df = pd.DataFrame(rows, columns=columns)
    return df


df_matches = extract_copy_table_from_sql(SQL_PATH, "matches")

print(df_matches.shape)
display(df_matches.head())

(25, 27)


,id,title,location,date,players_needed,description,image,created_at,duration,status,...,video_url,video_thumbnail_url,url_frame,client,cancelled_at,created_by,caller_name,caller_phone,homme_du_match,video_token
0,dc0f2af3-0753-4790-bbd0-87b7ed26b9e9,Match de ketata rym,Terrain 1,2026-06-04 09:30:00+00,10,Match organisé le jeudi 4 juin par ketata rym,\N,2026-06-04 08:07:00.870311+00,90,completed,...,\N,\N,\N,\N,\N,914860f7-4ef5-47cc-bcbe-4b4088cc89a4,\N,\N,\N,\N
1,bee4b346-9c3b-431f-a827-6f404888f191,Match de fififab,Terrain 2,2026-06-20 18:00:00+00,10,Match organisé le samedi 20 juin par fififab,\N,2026-06-17 09:36:49.690629+00,120,upcoming,...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
2,d702f6ac-a302-4af5-b16f-d9c7521e9afb,Match de fififab,Terrain 1,2026-06-17 09:13:18.696+00,10,Match organisé le mercredi 17 juin par fififab,\N,2026-06-17 09:33:46.102621+00,90,completed,...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
3,ced08a3d-8ffa-4c45-b6f3-19a9fb6c0080,Match de belghithfarah,Terrain 2,2026-06-19 18:00:00+00,10,Match organisé le vendredi 19 juin par belghit...,\N,2026-06-17 14:49:55.812436+00,90,upcoming,...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N
4,609ce802-6e05-4cb4-8a7b-f45159af7faf,Match de belghithfarah,Terrain 1,2026-06-26 20:30:00+00,10,Match organisé le vendredi 26 juin par belghit...,\N,2026-06-17 14:50:27.743359+00,120,upcoming,...,\N,\N,\N,\N,\N,\N,\N,\N,\N,\N


In [ ]:
df = df_matches.copy()

# Nettoyage des valeurs PostgreSQL nulles
df = df.replace("\\N", np.nan)

# Conversion date
df["date"] = pd.to_datetime(df["date"], errors="coerce")

# Garder seulement les lignes avec date valide
df = df.dropna(subset=["date"]).copy()

# Exclure les matchs annulés si la colonne status existe
if "status" in df.columns:
    df = df[df["status"].fillna("").str.lower() != "cancelled"].copy()

# Extraire date simple et heure
df["date_jour"] = df["date"].dt.date
df["heure_num"] = df["date"].dt.hour

# Compter les réservations existantes par date et heure
df_counts = (
    df.groupby(["date_jour", "heure_num"])
    .size()
    .reset_index(name="reservations")
)

# Définir la période complète
date_min = pd.to_datetime(df_counts["date_jour"]).min()
date_max = pd.to_datetime(df_counts["date_jour"]).max()

# Créer toutes les dates entre min et max
all_dates = pd.date_range(start=date_min, end=date_max, freq="D")

# Créer toutes les heures de la journée
all_hours = range(8, 24)

# Construire une grille complète date x heure
df_grid = pd.MultiIndex.from_product(
    [all_dates.date, all_hours],
    names=["date_jour", "heure_num"]
).to_frame(index=False)

# Fusionner avec les réservations réelles
df_reservations = df_grid.merge(
    df_counts,
    on=["date_jour", "heure_num"],
    how="left"
)

# Remplacer les heures sans réservation par 0
df_reservations["reservations"] = df_reservations["reservations"].fillna(0).astype(int)

# Convertir date
df_reservations["date"] = pd.to_datetime(df_reservations["date_jour"])

# Supprimer colonne intermédiaire
df_reservations = df_reservations.drop(columns=["date_jour"])

display(df_reservations.head(30))
print(df_reservations.shape)
print(df_reservations["reservations"].value_counts().sort_index())

/tmp/ipykernel_1712/776727631.py:4: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df = df.replace("\\N", np.nan)


,heure_num,reservations,date
0,8,0,2026-06-04
1,9,1,2026-06-04
2,10,0,2026-06-04
3,11,0,2026-06-04
4,12,0,2026-06-04
5,13,0,2026-06-04
6,14,0,2026-06-04
7,15,0,2026-06-04
8,16,0,2026-06-04
9,17,0,2026-06-04


(432, 3)
reservations
0    410
1     22
Name: count, dtype: int64


In [ ]:
def month_to_season_fr(month):
    if month in [12, 1, 2]:
        return "hiver"
    elif month in [3, 4, 5]:
        return "printemps"
    elif month in [6, 7, 8]:
        return "été"
    else:
        return "automne"


def partie_jour(h):
    if 5 <= h < 12:
        return "matin"
    elif 12 <= h < 18:
        return "apres_midi"
    elif 18 <= h < 22:
        return "soir"
    else:
        return "nuit"


def add_time_features(df):
    df = df.copy()

    df["date"] = pd.to_datetime(df["date"], errors="coerce")

    df["annee"] = df["date"].dt.year
    df["mois"] = df["date"].dt.month
    df["semaine_annee"] = df["date"].dt.isocalendar().week.astype(int)
    df["trimestre"] = df["date"].dt.quarter

    df["jour_semaine_num"] = df["date"].dt.dayofweek
    jours = {
        0: "lundi",
        1: "mardi",
        2: "mercredi",
        3: "jeudi",
        4: "vendredi",
        5: "samedi",
        6: "dimanche"
    }
    df["jour_semaine"] = df["jour_semaine_num"].map(jours)

    df["est_weekend"] = df["jour_semaine_num"].isin([5, 6]).astype(int)

    df["heure_sin"] = np.sin(2 * np.pi * df["heure_num"] / 24)
    df["heure_cos"] = np.cos(2 * np.pi * df["heure_num"] / 24)

    jour_annee = df["date"].dt.dayofyear
    df["jour_annee_sin"] = np.sin(2 * np.pi * jour_annee / 365)
    df["jour_annee_cos"] = np.cos(2 * np.pi * jour_annee / 365)

    df["saison"] = df["mois"].apply(month_to_season_fr)
    df["partie_jour"] = df["heure_num"].apply(partie_jour)

    df["is_peak_hour"] = df["heure_num"].isin([8, 9, 12, 13, 18, 19, 20]).astype(int)
    df["is_commute_hour"] = df["heure_num"].isin([7, 8, 9, 17, 18, 19]).astype(int)

    return df


df_model = add_time_features(df_reservations)

display(df_model.head())

,heure_num,reservations,date,annee,mois,semaine_annee,trimestre,jour_semaine_num,jour_semaine,est_weekend,heure_sin,heure_cos,jour_annee_sin,jour_annee_cos,saison,partie_jour,is_peak_hour,is_commute_hour
0,8,0,2026-06-04,2026,6,23,2,3,jeudi,0,8.660254e-01,-0.500000,0.455907,-0.890028,été,matin,1,1
1,9,1,2026-06-04,2026,6,23,2,3,jeudi,0,7.071068e-01,-0.707107,0.455907,-0.890028,été,matin,1,1
2,10,0,2026-06-04,2026,6,23,2,3,jeudi,0,5.000000e-01,-0.866025,0.455907,-0.890028,été,matin,0,0
3,11,0,2026-06-04,2026,6,23,2,3,jeudi,0,2.588190e-01,-0.965926,0.455907,-0.890028,été,matin,0,0
4,12,0,2026-06-04,2026,6,23,2,3,jeudi,0,1.224647e-16,-1.000000,0.455907,-0.890028,été,apres_midi,1,0


In [ ]:
FEATURES_SANS_SITE_ID = [
    "heure_num",
    "heure_sin",
    "heure_cos",
    "jour_semaine",
    "partie_jour",
    "est_weekend",
    "is_peak_hour",
    "is_commute_hour",
    "mois",
    "semaine_annee",
    "trimestre",
    "jour_annee_sin",
    "jour_annee_cos",
    "saison",
    "ferie",
    "jour_ouvrable",
    "is_vacances_scolaires",
    "meteo",
    "meteo_severe",
    "temperature_c",
    "humidite_pct",
    "vent_kmh",
    "event_type",
    "event_strength",
    "temp_froide",
    "temp_chaude",
    "humidite_forte",
    "vent_fort",
    "annee"
]

colonnes_finales = ["date", "reservations"] + FEATURES_SANS_SITE_ID

df_final = df_model[colonnes_finales].copy()

display(df_final.head())
print(df_final.shape)

,date,reservations,heure_num,heure_sin,heure_cos,jour_semaine,partie_jour,est_weekend,is_peak_hour,is_commute_hour,...,temperature_c,humidite_pct,vent_kmh,event_type,event_strength,temp_froide,temp_chaude,humidite_forte,vent_fort,annee
0,2026-06-04,0,8,8.660254e-01,-0.500000,jeudi,matin,0,1,1,...,20,60,10,none,0,0,0,0,0,2026
1,2026-06-04,1,9,7.071068e-01,-0.707107,jeudi,matin,0,1,1,...,20,60,10,none,0,0,0,0,0,2026
2,2026-06-04,0,10,5.000000e-01,-0.866025,jeudi,matin,0,0,0,...,20,60,10,none,0,0,0,0,0,2026
3,2026-06-04,0,11,2.588190e-01,-0.965926,jeudi,matin,0,0,0,...,20,60,10,none,0,0,0,0,0,2026
4,2026-06-04,0,12,1.224647e-16,-1.000000,jeudi,apres_midi,0,1,0,...,20,60,10,none,0,0,0,0,0,2026


(432, 31)


In [ ]:
df_final.to_csv(OUTPUT_MODEL, index=False, encoding="utf-8-sig")

print("CSV sauvegardé ici :", OUTPUT_MODEL)

CSV sauvegardé ici : /content/drive/MyDrive/PFE/donnees_pretes_modele.csv


In [ ]:
import os

print("Fichier final existe ?", os.path.exists(OUTPUT_MODEL))
print("Chemin :", OUTPUT_MODEL)

df_check = pd.read_csv(OUTPUT_MODEL)
print(df_check.shape)
display(df_check.head())
display(df_check.info())


Fichier final existe ? True
Chemin : /content/drive/MyDrive/PFE/donnees_pretes_modele.csv
(432, 31)


,date,reservations,heure_num,heure_sin,heure_cos,jour_semaine,partie_jour,est_weekend,is_peak_hour,is_commute_hour,...,temperature_c,humidite_pct,vent_kmh,event_type,event_strength,temp_froide,temp_chaude,humidite_forte,vent_fort,annee
0,2026-06-04,0,8,8.660254e-01,-0.500000,jeudi,matin,0,1,1,...,20,60,10,none,0,0,0,0,0,2026
1,2026-06-04,1,9,7.071068e-01,-0.707107,jeudi,matin,0,1,1,...,20,60,10,none,0,0,0,0,0,2026
2,2026-06-04,0,10,5.000000e-01,-0.866025,jeudi,matin,0,0,0,...,20,60,10,none,0,0,0,0,0,2026
3,2026-06-04,0,11,2.588190e-01,-0.965926,jeudi,matin,0,0,0,...,20,60,10,none,0,0,0,0,0,2026
4,2026-06-04,0,12,1.224647e-16,-1.000000,jeudi,apres_midi,0,1,0,...,20,60,10,none,0,0,0,0,0,2026


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 432 entries, 0 to 431
Data columns (total 31 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   date                   432 non-null    object 
 1   reservations           432 non-null    int64  
 2   heure_num              432 non-null    int64  
 3   heure_sin              432 non-null    float64
 4   heure_cos              432 non-null    float64
 5   jour_semaine           432 non-null    object 
 6   partie_jour            432 non-null    object 
 7   est_weekend            432 non-null    int64  
 8   is_peak_hour           432 non-null    int64  
 9   is_commute_hour        432 non-null    int64  
 10  mois                   432 non-null    int64  
 11  semaine_annee          432 non-null    int64  
 12  trimestre              432 non-null    int64  
 13  jour_annee_sin         432 non-null    float64
 14  jour_annee_cos         432 non-null    float64
 15  saison

None

In [ ]:
df_matches.to_csv(OUTPUT_RAW, index=False, encoding="utf-8-sig")
print("Fichier brut sauvegardé ici :", OUTPUT_RAW)

Fichier brut sauvegardé ici : /content/drive/MyDrive/PFE/matches_extracted.csv


In [ ]:
print("Colonnes finales :")
print(df_final["reservations"].value_counts().sort_index())


print("Nombre de colonnes :", len(df_final.columns))
print("Nombre de lignes :", len(df_final))

Colonnes finales :
reservations
0    410
1     22
Name: count, dtype: int64
Nombre de colonnes : 31
Nombre de lignes : 432


In [ ]:
LATITUDE = 48.8566
LONGITUDE = 2.3522
TIMEZONE = "Europe/Paris"

ZONE_FERIES = "metropole"
ZONE_SCOLAIRE = "C"
DEPARTEMENT = "Paris"
VILLE = None

BIG_EVENT_KEYWORDS = [
    "concert", "festival", "spectacle", "show",
    "champion", "champions league", "ligue des champions",
    "coupe", "cup", "finale", "tournoi", "match",
    "euro", "world cup", "coupe du monde",
    "olympique", "jo", "jeu olympique", "jeux olympiques"
]

In [ ]:
def safe_requests_get(url, params=None, timeout=30, retries=3, sleep_time=1):
    last_error = None

    for attempt in range(retries):
        try:
            response = requests.get(url, params=params, timeout=timeout)
            return response
        except Exception as e:
            last_error = e
            print(f"Tentative {attempt + 1}/{retries} échouée :", e)
            time.sleep(sleep_time)

    raise last_error

In [ ]:
def charger_jours_feries_france(zone="metropole"):
    url = f"https://etalab.github.io/jours-feries-france-data/json/{zone}.json"

    response = safe_requests_get(url, timeout=30)
    response.raise_for_status()

    data = response.json()

    feries = pd.DataFrame(list(data.items()), columns=["date_ferie", "nom_jour_ferie"])
    feries["date_ferie"] = pd.to_datetime(feries["date_ferie"], errors="coerce").dt.normalize()
    feries["ferie"] = 1

    return feries


def ajouter_jours_feries(df):
    df = df.copy()

    df = df.drop(
        columns=[c for c in ["ferie", "nom_jour_ferie", "jour_ouvrable"] if c in df.columns],
        errors="ignore"
    )

    if "jour_semaine_num" not in df.columns:
        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["jour_semaine_num"] = df["date"].dt.weekday

    feries = charger_jours_feries_france(ZONE_FERIES)

    df["date_norm"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

    df = df.merge(
        feries[["date_ferie", "ferie", "nom_jour_ferie"]],
        how="left",
        left_on="date_norm",
        right_on="date_ferie"
    )

    df["ferie"] = df["ferie"].fillna(0).astype(int)
    df["nom_jour_ferie"] = df["nom_jour_ferie"].fillna("Aucun")

    df["jour_ouvrable"] = ((df["jour_semaine_num"] < 5) & (df["ferie"] == 0)).astype(int)

    df = df.drop(columns=["date_norm", "date_ferie"], errors="ignore")

    return df

In [ ]:
def charger_vacances_scolaires_officielles(zone="C"):
    base_url = "https://data.education.gouv.fr/api/explore/v2.1/catalog/datasets/fr-en-calendrier-scolaire/records"

    all_rows = []
    offset = 0
    limit = 100

    while True:
        params = {
            "limit": limit,
            "offset": offset,
            "select": "description,start_date,end_date,zones,population"
        }

        resp = safe_requests_get(base_url, params=params, timeout=30)
        resp.raise_for_status()

        data = resp.json()
        results = data.get("results", [])

        if not results:
            break

        all_rows.extend(results)

        total_count = data.get("total_count", len(all_rows))
        offset += limit

        if offset >= total_count:
            break

    if not all_rows:
        return pd.DataFrame(columns=["description", "start_date", "end_date"])

    vac = pd.DataFrame(all_rows)

    for c in ["description", "start_date", "end_date", "zones", "population"]:
        if c not in vac.columns:
            vac[c] = None

    vac["population"] = vac["population"].astype(str).str.lower().str.strip()
    vac = vac[~vac["population"].str.contains("enseignant", na=False)].copy()

    vac["start_date"] = pd.to_datetime(
        vac["start_date"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None).dt.normalize()

    vac["end_date"] = pd.to_datetime(
        vac["end_date"],
        errors="coerce",
        utc=True
    ).dt.tz_convert(None).dt.normalize()

    vac = vac.dropna(subset=["start_date", "end_date"]).copy()

    vac["zones"] = vac["zones"].astype(str).str.lower().str.strip()
    vac["description"] = vac["description"].astype(str).str.lower().str.strip()

    zone_label = f"zone {str(zone).lower()}"

    mask_zone = (
        vac["zones"].str.contains(zone_label, na=False) |
        vac["zones"].str.contains("toutes les zones", na=False) |
        vac["zones"].str.contains("paris|versailles|créteil|creteil", na=False) |
        vac["description"].str.contains("noël|noel|été|ete", na=False)
    )

    vac = vac[mask_zone].copy()

    mots_cles = ["vacances", "toussaint", "noël", "noel", "hiver", "printemps", "été", "ete"]
    pattern = "|".join(mots_cles)

    vac = vac[vac["description"].str.contains(pattern, na=False)].copy()

    return vac[["description", "start_date", "end_date"]]


def ajouter_vacances_scolaires(df):
    df = df.copy()

    vacances = charger_vacances_scolaires_officielles(zone=ZONE_SCOLAIRE)

    df["date_norm"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()
    df["is_vacances_scolaires"] = 0

    for _, row in vacances.iterrows():
        mask = (df["date_norm"] >= row["start_date"]) & (df["date_norm"] <= row["end_date"])
        df.loc[mask, "is_vacances_scolaires"] = 1

    df = df.drop(columns=["date_norm"], errors="ignore")

    return df

In [ ]:
def map_weather_code(code):
    if pd.isna(code):
        return "nuageux"

    code = int(code)

    if code in [0, 1]:
        return "ensoleillé"
    elif code in [2, 3, 45, 48]:
        return "nuageux"
    elif code in [51, 53, 55, 56, 57, 61, 63, 65, 66, 67, 80, 81, 82]:
        return "pluvieux"
    elif code in [71, 73, 75, 77, 85, 86]:
        return "pluvieux"
    elif code in [95, 96, 99]:
        return "orageux"
    else:
        return "nuageux"


def charger_meteo_periode(date_start, date_end, lat=LATITUDE, lon=LONGITUDE, timezone=TIMEZONE):
    date_start = pd.to_datetime(date_start).normalize()
    date_end = pd.to_datetime(date_end).normalize()
    today = pd.Timestamp.today().normalize()

    all_meteo = []

    # Partie passée / aujourd'hui : archive
    if date_start <= today:
        past_start = date_start
        past_end = min(date_end, today)

        url_archive = "https://archive-api.open-meteo.com/v1/archive"

        params_archive = {
            "latitude": lat,
            "longitude": lon,
            "start_date": past_start.strftime("%Y-%m-%d"),
            "end_date": past_end.strftime("%Y-%m-%d"),
            "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
            "timezone": timezone
        }

        try:
            resp = safe_requests_get(url_archive, params=params_archive, timeout=60)
            resp.raise_for_status()
            data = resp.json()

            hourly = data.get("hourly", {})

            if hourly:
                df_past = pd.DataFrame({
                    "datetime": pd.to_datetime(hourly["time"]),
                    "temperature_c": hourly["temperature_2m"],
                    "humidite_pct": hourly["relative_humidity_2m"],
                    "vent_kmh": hourly["wind_speed_10m"],
                    "weather_code": hourly["weather_code"]
                })

                all_meteo.append(df_past)

        except Exception as e:
            print("Erreur météo historique :", e)

    # Partie future : forecast
    if date_end > today:
        future_start = max(date_start, today + pd.Timedelta(days=1))
        future_end = date_end

        # Open-Meteo forecast couvre seulement les jours futurs proches
        # On limite à 16 jours après aujourd'hui
        max_forecast_end = today + pd.Timedelta(days=16)
        future_end = min(future_end, max_forecast_end)

        if future_start <= future_end:
            url_forecast = "https://api.open-meteo.com/v1/forecast"

            params_forecast = {
                "latitude": lat,
                "longitude": lon,
                "start_date": future_start.strftime("%Y-%m-%d"),
                "end_date": future_end.strftime("%Y-%m-%d"),
                "hourly": "temperature_2m,relative_humidity_2m,wind_speed_10m,weather_code",
                "timezone": timezone
            }

            try:
                resp = safe_requests_get(url_forecast, params=params_forecast, timeout=60)
                resp.raise_for_status()
                data = resp.json()

                hourly = data.get("hourly", {})

                if hourly:
                    df_future = pd.DataFrame({
                        "datetime": pd.to_datetime(hourly["time"]),
                        "temperature_c": hourly["temperature_2m"],
                        "humidite_pct": hourly["relative_humidity_2m"],
                        "vent_kmh": hourly["wind_speed_10m"],
                        "weather_code": hourly["weather_code"]
                    })

                    all_meteo.append(df_future)

            except Exception as e:
                print("Erreur météo prévisionnelle :", e)

    if not all_meteo:
        return pd.DataFrame(columns=[
            "date_norm", "heure_num", "meteo",
            "temperature_c", "humidite_pct", "vent_kmh"
        ])

    df_meteo = pd.concat(all_meteo, ignore_index=True)

    df_meteo["date_norm"] = df_meteo["datetime"].dt.normalize()
    df_meteo["heure_num"] = df_meteo["datetime"].dt.hour
    df_meteo["meteo"] = df_meteo["weather_code"].apply(map_weather_code)

    df_meteo = df_meteo[
        ["date_norm", "heure_num", "meteo", "temperature_c", "humidite_pct", "vent_kmh"]
    ].copy()

    return df_meteo


def ajouter_meteo(df):
    df = df.copy()

    date_start = pd.to_datetime(df["date"], errors="coerce").min()
    date_end = pd.to_datetime(df["date"], errors="coerce").max()

    print("Téléchargement météo de", date_start.date(), "à", date_end.date())

    df_meteo = charger_meteo_periode(date_start, date_end)

    df["date_norm"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

    df = df.merge(
        df_meteo,
        how="left",
        on=["date_norm", "heure_num"]
    )

    df["meteo"] = df["meteo"].fillna("nuageux")
    df["temperature_c"] = pd.to_numeric(df["temperature_c"], errors="coerce").fillna(20)
    df["humidite_pct"] = pd.to_numeric(df["humidite_pct"], errors="coerce").fillna(60)
    df["vent_kmh"] = pd.to_numeric(df["vent_kmh"], errors="coerce").fillna(10)

    df["meteo_severe"] = df["meteo"].isin(["pluvieux", "orageux"]).astype(int)

    df["temp_froide"] = (df["temperature_c"] < 8).astype(int)
    df["temp_chaude"] = (df["temperature_c"] > 28).astype(int)
    df["humidite_forte"] = (df["humidite_pct"] > 80).astype(int)
    df["vent_fort"] = (df["vent_kmh"] > 35).astype(int)

    df = df.drop(columns=["date_norm"], errors="ignore")

    return df

In [ ]:
def charger_gros_evenements_locaux(date_start, date_end, departement=None, ville=None, limit=100):
    base_url = "https://public.opendatasoft.com/api/explore/v2.1/catalog/datasets/evenements-publics-openagenda/records"

    start = pd.Timestamp(date_start).normalize()
    end = pd.Timestamp(date_end).normalize()

    all_rows = []

    for day in pd.date_range(start=start, end=end, freq="D"):
        day_start = day.normalize()
        day_end = day_start + pd.Timedelta(days=1)

        conditions = [
            f"firstdate_begin < date'{day_end:%Y-%m-%d}T00:00:00+00:00'",
            f"lastdate_end >= date'{day_start:%Y-%m-%d}T00:00:00+00:00'"
        ]

        if departement:
            dep = str(departement).replace("'", "''")
            conditions.append(f"location_department = '{dep}'")

        if ville:
            v = str(ville).replace("'", "''")
            conditions.append(f"location_city = '{v}'")

        where_clause = " AND ".join(conditions)

        offset = 0

        while True:
            params = {
                "select": "title_fr,keywords_fr,firstdate_begin,lastdate_end,timings,location_city,location_department",
                "where": where_clause,
                "limit": limit,
                "offset": offset,
                "order_by": "firstdate_begin"
            }

            try:
                resp = safe_requests_get(base_url, params=params, timeout=30)
                resp.raise_for_status()
                data = resp.json()
            except Exception as e:
                print("Erreur API événements :", e)
                break

            results = data.get("results", [])

            if not results:
                break

            all_rows.extend(results)

            total_count = data.get("total_count", 0)
            offset += limit

            if offset >= total_count:
                break

            if offset + limit > 10000:
                break

    events = pd.DataFrame(all_rows)

    if events.empty:
        return pd.DataFrame(columns=["date_event_local", "nom_event_local"])

    events["title_fr"] = events["title_fr"].fillna("").astype(str)
    events["keywords_fr"] = events["keywords_fr"].fillna("").astype(str)

    if BIG_EVENT_KEYWORDS:
        pattern = "|".join([kw.lower().replace("+", r"\+") for kw in BIG_EVENT_KEYWORDS])

        filtered = events[
            events["title_fr"].str.lower().str.contains(pattern, na=False, regex=True) |
            events["keywords_fr"].str.lower().str.contains(pattern, na=False, regex=True)
        ].copy()

        if not filtered.empty:
            events = filtered

    events["firstdate_begin"] = pd.to_datetime(events["firstdate_begin"], errors="coerce", utc=True)
    events["lastdate_end"] = pd.to_datetime(events["lastdate_end"], errors="coerce", utc=True)

    expanded_rows = []

    for _, row in events.iterrows():
        first_day = row["firstdate_begin"]
        last_day = row["lastdate_end"]

        if pd.isna(first_day):
            continue

        first_day = first_day.tz_convert(None).normalize()

        if pd.isna(last_day):
            last_day = first_day
        else:
            last_day = last_day.tz_convert(None).normalize()

        for target_day in pd.date_range(first_day, last_day, freq="D"):
            if start <= target_day <= end:
                expanded_rows.append({
                    "date_event_local": target_day.normalize(),
                    "nom_event_local": row["title_fr"]
                })

    events_expanded = pd.DataFrame(expanded_rows)

    if events_expanded.empty:
        return pd.DataFrame(columns=["date_event_local", "nom_event_local"])

    events_grouped = events_expanded.groupby("date_event_local", as_index=False).agg({
        "nom_event_local": lambda x: " | ".join(sorted(set([str(i) for i in x if str(i).strip() != ""])))
    })

    return events_grouped


def ajouter_evenements_locaux(df):
    df = df.copy()

    date_start = pd.to_datetime(df["date"], errors="coerce").min().strftime("%Y-%m-%d")
    date_end = pd.to_datetime(df["date"], errors="coerce").max().strftime("%Y-%m-%d")

    events = charger_gros_evenements_locaux(
        date_start=date_start,
        date_end=date_end,
        departement=DEPARTEMENT,
        ville=VILLE
    )

    df["date_norm"] = pd.to_datetime(df["date"], errors="coerce").dt.normalize()

    df = df.merge(
        events,
        how="left",
        left_on="date_norm",
        right_on="date_event_local"
    )

    df["nom_event_local"] = df["nom_event_local"].fillna("Aucun")

    df["event_type"] = np.where(
        df["nom_event_local"] != "Aucun",
        "event_local",
        "none"
    )

    df["event_strength"] = np.where(
        df["nom_event_local"] != "Aucun",
        2,
        0
    )

    df = df.drop(columns=["date_norm", "date_event_local"], errors="ignore")

    return df

In [ ]:
df_model = add_time_features(df_reservations)

display(df_model.head())

,heure_num,reservations,date,annee,mois,semaine_annee,trimestre,jour_semaine_num,jour_semaine,est_weekend,heure_sin,heure_cos,jour_annee_sin,jour_annee_cos,saison,partie_jour,is_peak_hour,is_commute_hour
0,8,0,2026-06-04,2026,6,23,2,3,jeudi,0,8.660254e-01,-0.500000,0.455907,-0.890028,été,matin,1,1
1,9,1,2026-06-04,2026,6,23,2,3,jeudi,0,7.071068e-01,-0.707107,0.455907,-0.890028,été,matin,1,1
2,10,0,2026-06-04,2026,6,23,2,3,jeudi,0,5.000000e-01,-0.866025,0.455907,-0.890028,été,matin,0,0
3,11,0,2026-06-04,2026,6,23,2,3,jeudi,0,2.588190e-01,-0.965926,0.455907,-0.890028,été,matin,0,0
4,12,0,2026-06-04,2026,6,23,2,3,jeudi,0,1.224647e-16,-1.000000,0.455907,-0.890028,été,apres_midi,1,0


In [ ]:
df_model = add_time_features(df_reservations)

print("Ajout des jours fériés...")
df_model = ajouter_jours_feries(df_model)

print("Ajout des vacances scolaires...")
df_model = ajouter_vacances_scolaires(df_model)

print("Ajout de la météo...")
df_model = ajouter_meteo(df_model)

print("Ajout des événements locaux...")
df_model = ajouter_evenements_locaux(df_model)

display(df_model.head())

Ajout des jours fériés...
Ajout des vacances scolaires...
Ajout de la météo...
Téléchargement météo de 2026-06-04 à 2026-06-30
Ajout des événements locaux...


,heure_num,reservations,date,annee,mois,semaine_annee,trimestre,jour_semaine_num,jour_semaine,est_weekend,...,humidite_pct,vent_kmh,meteo_severe,temp_froide,temp_chaude,humidite_forte,vent_fort,nom_event_local,event_type,event_strength
0,8,0,2026-06-04,2026,6,23,2,3,jeudi,0,...,83,17.7,1,0,0,1,0,"Atelier découpeuse vinyle | C3RV4U, l'expo neu...",event_local,2
1,9,1,2026-06-04,2026,6,23,2,3,jeudi,0,...,83,18.9,1,0,0,1,0,"Atelier découpeuse vinyle | C3RV4U, l'expo neu...",event_local,2
2,10,0,2026-06-04,2026,6,23,2,3,jeudi,0,...,82,17.1,1,0,0,1,0,"Atelier découpeuse vinyle | C3RV4U, l'expo neu...",event_local,2
3,11,0,2026-06-04,2026,6,23,2,3,jeudi,0,...,82,16.4,1,0,0,1,0,"Atelier découpeuse vinyle | C3RV4U, l'expo neu...",event_local,2
4,12,0,2026-06-04,2026,6,23,2,3,jeudi,0,...,69,20.1,1,0,0,0,0,"Atelier découpeuse vinyle | C3RV4U, l'expo neu...",event_local,2


In [ ]:
print("Répartition météo :")
print(df_model["meteo"].value_counts())

print("\nTempérature :")
print(df_model["temperature_c"].describe())

print("\nHumidité :")
print(df_model["humidite_pct"].describe())

print("\nVent :")
print(df_model["vent_kmh"].describe())


Répartition météo :
meteo
nuageux       189
ensoleillé    168
pluvieux       75
Name: count, dtype: int64

Température :
count    432.000000
mean      24.783796
std        7.335947
min       11.600000
25%       18.075000
50%       23.850000
75%       30.825000
max       41.100000
Name: temperature_c, dtype: float64

Humidité :
count    432.000000
mean      49.712963
std       16.407401
min       16.000000
25%       36.000000
50%       49.000000
75%       62.000000
max       91.000000
Name: humidite_pct, dtype: float64

Vent :
count    432.000000
mean       9.798843
std        4.523091
min        0.400000
25%        6.275000
50%        9.700000
75%       12.600000
max       22.800000
Name: vent_kmh, dtype: float64


In [ ]:
FEATURES_SANS_SITE_ID = [
    "heure_num",
    "heure_sin",
    "heure_cos",
    "jour_semaine",
    "partie_jour",
    "est_weekend",
    "is_peak_hour",
    "is_commute_hour",
    "mois",
    "semaine_annee",
    "trimestre",
    "jour_annee_sin",
    "jour_annee_cos",
    "saison",
    "ferie",
    "jour_ouvrable",
    "is_vacances_scolaires",
    "meteo",
    "meteo_severe",
    "temperature_c",
    "humidite_pct",
    "vent_kmh",
    "event_type",
    "event_strength",
    "temp_froide",
    "temp_chaude",
    "humidite_forte",
    "vent_fort",
    "annee"
]

colonnes_finales = ["date", "reservations"] + FEATURES_SANS_SITE_ID

df_final = df_model[colonnes_finales].copy()

display(df_final.head())
print(df_final.shape)

,date,reservations,heure_num,heure_sin,heure_cos,jour_semaine,partie_jour,est_weekend,is_peak_hour,is_commute_hour,...,temperature_c,humidite_pct,vent_kmh,event_type,event_strength,temp_froide,temp_chaude,humidite_forte,vent_fort,annee
0,2026-06-04,0,8,8.660254e-01,-0.500000,jeudi,matin,0,1,1,...,15.4,83,17.7,event_local,2,0,0,1,0,2026
1,2026-06-04,1,9,7.071068e-01,-0.707107,jeudi,matin,0,1,1,...,15.3,83,18.9,event_local,2,0,0,1,0,2026
2,2026-06-04,0,10,5.000000e-01,-0.866025,jeudi,matin,0,0,0,...,15.9,82,17.1,event_local,2,0,0,1,0,2026
3,2026-06-04,0,11,2.588190e-01,-0.965926,jeudi,matin,0,0,0,...,15.3,82,16.4,event_local,2,0,0,1,0,2026
4,2026-06-04,0,12,1.224647e-16,-1.000000,jeudi,apres_midi,0,1,0,...,17.9,69,20.1,event_local,2,0,0,0,0,2026


(432, 31)


In [ ]:
df_final.to_csv(OUTPUT_MODEL, index=False, encoding="utf-8-sig")

print("CSV sauvegardé ici :", OUTPUT_MODEL)

CSV sauvegardé ici : /content/drive/MyDrive/PFE/donnees_pretes_modele.csv


In [ ]:
import os

print("Fichier final existe ?", os.path.exists(OUTPUT_MODEL))
print("Chemin :", OUTPUT_MODEL)

df_check = pd.read_csv(OUTPUT_MODEL)

print(df_check.shape)
display(df_check.head())
print(df_check["reservations"])

Fichier final existe ? True
Chemin : /content/drive/MyDrive/PFE/donnees_pretes_modele.csv
(432, 31)


,date,reservations,heure_num,heure_sin,heure_cos,jour_semaine,partie_jour,est_weekend,is_peak_hour,is_commute_hour,...,temperature_c,humidite_pct,vent_kmh,event_type,event_strength,temp_froide,temp_chaude,humidite_forte,vent_fort,annee
0,2026-06-04,0,8,8.660254e-01,-0.500000,jeudi,matin,0,1,1,...,15.4,83,17.7,event_local,2,0,0,1,0,2026
1,2026-06-04,1,9,7.071068e-01,-0.707107,jeudi,matin,0,1,1,...,15.3,83,18.9,event_local,2,0,0,1,0,2026
2,2026-06-04,0,10,5.000000e-01,-0.866025,jeudi,matin,0,0,0,...,15.9,82,17.1,event_local,2,0,0,1,0,2026
3,2026-06-04,0,11,2.588190e-01,-0.965926,jeudi,matin,0,0,0,...,15.3,82,16.4,event_local,2,0,0,1,0,2026
4,2026-06-04,0,12,1.224647e-16,-1.000000,jeudi,apres_midi,0,1,0,...,17.9,69,20.1,event_local,2,0,0,0,0,2026


0      0
1      1
2      0
3      0
4      0
      ..
427    0
428    0
429    0
430    0
431    0
Name: reservations, Length: 432, dtype: int64
